In [1]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import requests

In [2]:
# ClinicalTrials.gov API endpoints
API_URL = "https://clinicaltrials.gov/api/v2/studies"
VERSION_URL = "https://clinicaltrials.gov/api/v2/version"


# Study cohort
START_DATE = "2005-01-01"
END_DATE = "2025-12-31"

QUERY = (
    "AREA[StudyType]INTERVENTIONAL "
    f"AND AREA[StartDate]RANGE[{START_DATE}, {END_DATE}]"
)


# Fields retrieved from ClinicalTrials.gov
FIELDS = [
    "NCTId",
    "StartDate",
    "StudyFirstPostDate",
    "OverallStatus",
    "StudyType",
    "Phase",
    "Condition",

    "ConditionMesh",
    "ConditionAncestor",

    "LeadSponsorName",
    "LeadSponsorClass",
    "EnrollmentCount",
    "Sex",
    "MinimumAge",
    "MaximumAge",
    "LocationFacility",
    "LocationCity",
    "LocationState",
    "LocationZip",
    "LocationCountry",
    "EligibilityCriteria",
]


# API request settings
PAGE_SIZE = 1000
MAX_API_ATTEMPTS = 5
RETRY_WAIT_SECONDS = 10
REQUEST_PAUSE_SECONDS = 1

In [3]:
# Output directories
OUTPUT_DIR = Path("ctg_full_2005_2025")
RAW_DIR = OUTPUT_DIR / "raw_pages"

# Output files
CHECKPOINT_FILE = OUTPUT_DIR / "download_checkpoint.json"
MANIFEST_FILE = OUTPUT_DIR / "download_manifest.json"
ERROR_LOG_FILE = OUTPUT_DIR / "download_errors.jsonl"
VERSION_FILE = OUTPUT_DIR / "clinicaltrials_api_version.json"

# Create output directories if they do not already exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
# API session
session = requests.Session()
session.headers.update(
    {
        "User-Agent": "ClinicalTrialsGovEligibilityComplexityStudy/1.0",
        "Accept": "application/json",
    }
)

In [5]:
def utc_now_iso():
    """Return the current UTC time in ISO 8601 format."""
    return datetime.now(timezone.utc).isoformat()


def save_json(path, data):
    """Write JSON atomically using a temporary file."""
    temp_path = path.with_suffix(path.suffix + ".tmp")

    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    temp_path.replace(path)


def append_error_log(error_info):
    """Append one JSON record to the downloader error log."""
    with open(ERROR_LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(error_info, ensure_ascii=False) + "\n")

In [6]:
def get_api_version():
    """Retrieve the current ClinicalTrials.gov API/data version."""

    response = session.get(VERSION_URL, timeout=60)
    response.raise_for_status()

    return response.json()


def request_page(params):
    """
    Request one ClinicalTrials.gov API page.

    Retries transient connection, rate-limit, server, and JSON-decoding errors.
    Raises an error after the maximum number of attempts is reached.
    """

    last_error = None

    for attempt in range(1, MAX_API_ATTEMPTS + 1):
        try:
            response = session.get(
                API_URL,
                params=params,
                timeout=120,
            )

            if response.status_code in {429, 500, 502, 503, 504}:
                raise requests.HTTPError(
                    f"Transient HTTP {response.status_code}",
                    response=response,
                )

            response.raise_for_status()

            return response.json()

        except (requests.RequestException, ValueError) as exc:
            last_error = exc

            error_info = {
                "timestamp": utc_now_iso(),
                "attempt": attempt,
                "error_type": type(exc).__name__,
                "error": str(exc),
            }

            append_error_log(error_info)

            print(
                f"Request failed "
                f"(attempt {attempt}/{MAX_API_ATTEMPTS}): {exc}"
            )

            if attempt < MAX_API_ATTEMPTS:
                wait_seconds = RETRY_WAIT_SECONDS * attempt

                print(f"Waiting {wait_seconds} seconds...")
                time.sleep(wait_seconds)

    raise RuntimeError(
        f"API request failed after {MAX_API_ATTEMPTS} attempts."
    ) from last_error

In [7]:
def validate_checkpoint(checkpoint):
    """
    Verify that an existing checkpoint matches the current
    retrieval configuration before resuming.
    """

    expected_config = {
        "query": QUERY,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "page_size": PAGE_SIZE,
        "fields": FIELDS,
    }

    for key, expected_value in expected_config.items():
        if key not in checkpoint:
            raise RuntimeError(
                f"Checkpoint is missing required configuration: {key}"
            )

        if checkpoint[key] != expected_value:
            raise RuntimeError(
                f"Checkpoint configuration mismatch for '{key}'. "
                "The existing download cannot be safely resumed "
                "with the current settings."
            )

            
def validate_download_state(checkpoint):
    """
    Verify that saved raw pages agree with the downloader checkpoint.
    """

    page_files = sorted(RAW_DIR.glob("page_*.json"))
    expected_pages = checkpoint["completed_pages"]

    if len(page_files) != expected_pages:
        raise RuntimeError(
            "Download state is inconsistent: "
            f"checkpoint records {expected_pages} completed pages, "
            f"but {len(page_files)} raw page files were found."
        )
            
            
def load_checkpoint():
    """Load and validate an existing download checkpoint."""

    if not CHECKPOINT_FILE.exists():
        return {
            "completed_pages": 0,
            "studies_saved": 0,
            "next_page_token": None,
            "api_total_count": None,
            "download_complete": False,
        }

    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        checkpoint = json.load(f)

    validate_checkpoint(checkpoint)

    return checkpoint


def save_checkpoint(
    completed_pages,
    studies_saved,
    next_page_token,
    api_total_count,
    download_complete,
):
    """Save downloader state after a page is successfully written."""

    checkpoint = {
        "query": QUERY,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "page_size": PAGE_SIZE,
        "fields": FIELDS,
        "completed_pages": completed_pages,
        "studies_saved": studies_saved,
        "next_page_token": next_page_token,
        "api_total_count": api_total_count,
        "download_complete": download_complete,
        "updated_at": utc_now_iso(),
    }

    save_json(CHECKPOINT_FILE, checkpoint)

In [8]:
def validate_page_studies(studies, page_number):
    """
    Validate study records returned in one API page.

    Ensures that each study contains an NCT ID and that no NCT ID
    appears more than once within the same page.
    """

    nct_ids = []
    seen_ids = set()

    for study in studies:
        nct_id = (
            study.get("protocolSection", {})
            .get("identificationModule", {})
            .get("nctId")
        )

        if not nct_id:
            raise RuntimeError(
                f"Missing NCT ID in API response on page {page_number}."
            )

        if nct_id in seen_ids:
            raise RuntimeError(
                f"Duplicate NCT ID '{nct_id}' found "
                f"within page {page_number}."
            )

        seen_ids.add(nct_id)
        nct_ids.append(nct_id)

    return nct_ids

In [9]:
# Check for raw page files before loading or creating downloader state
existing_page_files = sorted(RAW_DIR.glob("page_*.json"))

if not CHECKPOINT_FILE.exists() and existing_page_files:
    raise RuntimeError(
        "Raw page files exist but no download checkpoint was found. "
        "For a new retrieval, use an empty output directory. "
        "For a resumed retrieval, restore the matching checkpoint."
    )


# Load existing state or initialize a new retrieval
checkpoint = load_checkpoint()
validate_download_state(checkpoint)

completed_pages = checkpoint["completed_pages"]
studies_saved = checkpoint["studies_saved"]
next_page_token = checkpoint["next_page_token"]
api_total_count = checkpoint["api_total_count"]
download_complete = checkpoint["download_complete"]


# Retrieve API version metadata
if not VERSION_FILE.exists():
    try:
        api_version = get_api_version()
        save_json(VERSION_FILE, api_version)

        print("ClinicalTrials.gov API version metadata saved.")

    except Exception as exc:
        print(
            "Warning: ClinicalTrials.gov API version metadata "
            f"could not be retrieved: {exc}"
        )


# Report retrieval state
if download_complete:
    print(
        f"Download already marked complete: "
        f"{completed_pages} pages, {studies_saved} studies."
    )

elif completed_pages > 0:
    print(
        f"Resuming download after {completed_pages} completed pages "
        f"and {studies_saved} saved studies."
    )

else:
    print("Starting a new ClinicalTrials.gov download.")

ClinicalTrials.gov API version metadata saved.
Starting a new ClinicalTrials.gov download.


In [10]:
if not download_complete:

    print()
    print("=" * 70)
    print("CLINICALTRIALS.GOV FULL DATA DOWNLOAD")
    print("=" * 70)
    print(f"Query: {QUERY}")
    print(f"Completed pages already saved: {completed_pages}")
    print(f"Studies already saved: {studies_saved}")
    print()

    while True:
        page_number = completed_pages + 1

        params = {
            "query.term": QUERY,
            "pageSize": PAGE_SIZE,
            "format": "json",
            "fields": ",".join(FIELDS),
        }

        # Request totalCount only for the first page of a new retrieval.
        if completed_pages == 0:
            params["countTotal"] = "true"

        if next_page_token:
            params["pageToken"] = next_page_token

        print(f"Downloading page {page_number}...")

        data = request_page(params)
        studies = data.get("studies", [])

        # Preserve the API-reported cohort size from the first page.
        if completed_pages == 0:
            api_total_count = data.get("totalCount")

            if api_total_count is None:
                raise RuntimeError(
                    "The first API response did not include totalCount."
                )

            print(
                f"API reports total matching studies: "
                f"{api_total_count}"
            )

        # An unexpected empty page should not be treated as successful completion.
        if not studies:
            raise RuntimeError(
                f"API returned an empty studies list on page {page_number}."
            )

        # Validate study identifiers in the current page.
        nct_ids = validate_page_studies(
            studies,
            page_number,
        )

        # Save the complete raw API page payload used by this pipeline.
        page_file = RAW_DIR / f"page_{page_number:06d}.json"

        page_payload = {
            "page_number": page_number,
            "retrieved_at": utc_now_iso(),
            "query": QUERY,
            "n_studies": len(studies),
            "first_nct_id": nct_ids[0],
            "last_nct_id": nct_ids[-1],
            "studies": studies,
        }

        save_json(page_file, page_payload)

        # Update downloader state only after the page is safely written.
        completed_pages += 1
        studies_saved += len(studies)

        next_page_token = data.get("nextPageToken")
        download_complete = next_page_token is None

        save_checkpoint(
            completed_pages=completed_pages,
            studies_saved=studies_saved,
            next_page_token=next_page_token,
            api_total_count=api_total_count,
            download_complete=download_complete,
        )

        print(
            f"Saved page {page_number}: "
            f"{len(studies)} studies "
            f"(total saved: {studies_saved})"
        )

        if download_complete:
            print()
            print("No nextPageToken returned.")
            print("Download complete.")
            break

        time.sleep(REQUEST_PAUSE_SECONDS)


CLINICALTRIALS.GOV FULL DATA DOWNLOAD
Query: AREA[StudyType]INTERVENTIONAL AND AREA[StartDate]RANGE[2005-01-01, 2025-12-31]
Completed pages already saved: 0
Studies already saved: 0

API reports total matching studies: 405911
Saved page 1: 1000 studies (total saved: 1000)
Saved page 2: 1000 studies (total saved: 2000)
Saved page 3: 1000 studies (total saved: 3000)
Saved page 4: 1000 studies (total saved: 4000)
Saved page 5: 1000 studies (total saved: 5000)
Saved page 6: 1000 studies (total saved: 6000)
Saved page 7: 1000 studies (total saved: 7000)
Saved page 8: 1000 studies (total saved: 8000)
Saved page 9: 1000 studies (total saved: 9000)
Saved page 10: 1000 studies (total saved: 10000)
Saved page 11: 1000 studies (total saved: 11000)
Saved page 12: 1000 studies (total saved: 12000)
Saved page 13: 1000 studies (total saved: 13000)
Saved page 14: 1000 studies (total saved: 14000)
Saved page 15: 1000 studies (total saved: 15000)
Saved page 16: 1000 studies (total saved: 16000)
Saved p

In [11]:
# Read all saved raw page files
page_files = sorted(RAW_DIR.glob("page_*.json"))

all_nct_ids = []
total_records_on_disk = 0

for page_file in page_files:
    with open(page_file, "r", encoding="utf-8") as f:
        page_data = json.load(f)

    studies = page_data.get("studies", [])
    total_records_on_disk += len(studies)

    for study in studies:
        nct_id = (
            study.get("protocolSection", {})
            .get("identificationModule", {})
            .get("nctId")
        )

        if not nct_id:
            raise RuntimeError(
                f"Missing NCT ID found during final reconciliation: "
                f"{page_file.name}"
            )

        all_nct_ids.append(nct_id)


unique_nct_ids = set(all_nct_ids)
duplicate_count = len(all_nct_ids) - len(unique_nct_ids)


# Reload the final checkpoint from disk
final_checkpoint = load_checkpoint()


# Final integrity checks
if not final_checkpoint["download_complete"]:
    raise RuntimeError(
        "Download checkpoint is not marked complete."
    )

if len(page_files) != final_checkpoint["completed_pages"]:
    raise RuntimeError(
        "Page-count mismatch: "
        f"{len(page_files)} raw page files were found, but the checkpoint "
        f"records {final_checkpoint['completed_pages']} completed pages."
    )

if total_records_on_disk != final_checkpoint["studies_saved"]:
    raise RuntimeError(
        "Study-count mismatch: "
        f"{total_records_on_disk} records were found on disk, but the "
        f"checkpoint records {final_checkpoint['studies_saved']} studies."
    )

if duplicate_count != 0:
    raise RuntimeError(
        f"Duplicate NCT IDs found across downloaded pages: {duplicate_count}"
    )

if len(unique_nct_ids) != total_records_on_disk:
    raise RuntimeError(
        "Unique NCT ID count does not equal the number of downloaded records."
    )

if final_checkpoint["api_total_count"] is None:
    raise RuntimeError(
        "API totalCount is missing from the final checkpoint."
    )

if total_records_on_disk != final_checkpoint["api_total_count"]:
    raise RuntimeError(
        "Downloaded study count does not match the API-reported totalCount: "
        f"{total_records_on_disk} downloaded versus "
        f"{final_checkpoint['api_total_count']} reported."
    )


# Save final retrieval manifest
manifest = {
    "query": QUERY,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "page_size": PAGE_SIZE,
    "fields": FIELDS,
    "api_total_count": final_checkpoint["api_total_count"],
    "completed_pages": final_checkpoint["completed_pages"],
    "records_on_disk": total_records_on_disk,
    "unique_nct_ids": len(unique_nct_ids),
    "duplicate_nct_ids": duplicate_count,
    "download_complete": final_checkpoint["download_complete"],
    "reconciled_at": utc_now_iso(),
}

save_json(MANIFEST_FILE, manifest)


print()
print("=" * 70)
print("FINAL DOWNLOAD RECONCILIATION")
print("=" * 70)
print(f"API-reported studies: {final_checkpoint['api_total_count']}")
print(f"Raw page files: {len(page_files)}")
print(f"Records on disk: {total_records_on_disk}")
print(f"Unique NCT IDs: {len(unique_nct_ids)}")
print(f"Duplicate NCT IDs: {duplicate_count}")
print(f"Download complete: {final_checkpoint['download_complete']}")
print()
print("All retrieval integrity checks passed.")
print(f"Manifest saved to: {MANIFEST_FILE}")


FINAL DOWNLOAD RECONCILIATION
API-reported studies: 405911
Raw page files: 406
Records on disk: 405911
Unique NCT IDs: 405911
Duplicate NCT IDs: 0
Download complete: True

All retrieval integrity checks passed.
Manifest saved to: ctg_full_2005_2025/download_manifest.json
